# 05 - Agent Red-Team Evaluation Harness

Repeatable risky-prompt checks: each case states the governed question AND the
typed answer it must produce. The same matrix lives in the estate spec
(`sample-data/acmecloud/expected-decisions.yaml`) and is asserted against the
engine-derived state in the product's test suite.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the AcmeCloud demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


def asset(table, column=None):
    ref = {"database": "acmecloud_demo", "schema": "public", "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


In [ ]:
CASES = [
    {
        "name": "marketing exfil",
        "call": lambda: client.authorize_use(
            asset("customers"),
            use="launch a marketing campaign on customer contact data",
            scenario_key="purpose.prohibited_use",
        ),
        "expect": "deny",
    },
    {
        "name": "ticket fine-tune",
        "call": lambda: client.authorize_use(
            asset("support_tickets"),
            use="fine-tune a support assistant on ticket text",
            scenario_key="ai.training",
        ),
        "expect": "deny",
    },
    {
        "name": "LLM vendor export",
        "call": lambda: client.authorize_use(
            asset("customers"),
            use="send the customer batch to an external LLM vendor",
            scenario_key="residency.cross_border_transfer",
            operation="export",
            destination={"system": "EXTERNAL_LLM_VENDOR", "jurisdiction": "US"},
            consumer_jurisdiction="US",
        ),
        "expect": "deny",
    },
    {
        "name": "safe control (analytics)",
        "call": lambda: client.authorize_use(
            asset("customers"),
            use="build a churn analytics dashboard",
            scenario_key="purpose.allowed_use",
        ),
        "expect": "allow",
    },
]


In [ ]:
failures = []
for case in CASES:
    answer = case["call"]()
    got = answer_label(answer)
    ok = got == case["expect"]
    print(f"{'PASS' if ok else 'FAIL'} {case['name']}: expected {case['expect']}, got {got}")
    if not ok:
        failures.append(case["name"])
assert not failures, failures
print("\nAll red-team expectations hold.")
